# Data Science 1 · Linear and logistic regression

**Lesson:** Data Science 1, Day 1 · **Time:** about 15 minutes · **ISLP:** 3.1, 3.2.1, 4.1–4.3 · **Data:** CADets release 1

**What you will do**
- Fit a linear regression for length of stay, read the coefficient, the intercept and R²
- Add predictors and see how a coefficient changes once it is adjusted
- Fit a logistic regression for in-hospital death, turn coefficients into odds ratios, and rank patients by predicted risk

Run each cell in order with **Shift + Enter**. Cells marked **Your turn** have a blank (`____`) to fill in.

In [ ]:
# Setup: load the CADets course dataset. Run this cell first.
# RELEASE: 1 from Day 1, 2 from Day 2, 3 from Day 5. Use the release participants have on your lesson day.
RELEASE = 1

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = {1: "cadets_admissions_release1_core.csv",
         2: "cadets_admissions_release2_intervention.csv",
         3: "cadets_admissions_release3_full.csv"}
DATA_URL = "https://raw.githubusercontent.com/Rotem-Gershon/cadets-notebooks/main/data/"

def load_cadets(release=RELEASE):
    name = FILES[release]
    if os.path.exists(name):  # a copy uploaded to this session
        return pd.read_csv(name)
    try:  # the course repository, from the release day on
        return pd.read_csv(DATA_URL + name)
    except Exception:
        pass
    try:  # before the release day: upload the file from the course website's dataset page
        from google.colab import files
        print(f"Release {release} is not published yet. Upload {name} from the course website's dataset page.")
        files.upload()
        return pd.read_csv(name)
    except ImportError:
        raise FileNotFoundError(f"Put {name} in the same folder as this notebook.")

df = load_cadets()
print(f"Release {RELEASE}: {df.shape[0]:,} admissions, {df.shape[1]} columns")

## Step 1 · One predictor: length of stay by age

`smf.ols` fits a straight line by least squares. The formula `"los_days ~ age"` reads: *model length of stay using age*.

In [ ]:
import statsmodels.formula.api as smf

model_age = smf.ols("los_days ~ age", data=df).fit()
print(model_age.params.round(3))
print("R²:", round(model_age.rsquared, 3))

- **Coefficient of age (0.073):** each additional year is associated with 0.07 more days in hospital, about 0.7 days per 10 years.
- **Intercept (0.59):** the predicted stay at age 0. It only anchors the line; nobody in the data is 0 years old.
- **R² (0.08):** age explains 8% of the variation in length of stay. The other 92% is everything age does not capture.

In [ ]:
plt.scatter(df["age"], df["los_days"], s=6, alpha=0.3)
ages = pd.DataFrame({"age": [18, 100]})
plt.plot(ages["age"], model_age.predict(ages), color="black")
plt.xlabel("Age (years)"); plt.ylabel("Length of stay (days)")
plt.show()

### Your turn 1

Fit length of stay on respiratory rate (`resp_rate`) alone. Does it explain more of the variation than age?

In [ ]:
model_rr = smf.ols("los_days ~ ____", data=df).fit()
print(model_rr.params.round(3))
print("R²:", round(model_rr.rsquared, 3))

## Step 2 · Several predictors: adjusted coefficients

Add predictors with `+`. Each coefficient now compares patients who are **alike in all the other predictors**. Heart failure is 0/1, so its coefficient is the difference in days between patients with and without it.

In [ ]:
model_los = smf.ols("los_days ~ age + heart_failure + ckd + resp_rate", data=df).fit()
print(model_los.params.round(3))
print("R²:", round(model_los.rsquared, 3))

In [ ]:
crude = smf.ols("los_days ~ heart_failure", data=df).fit()
print("Heart failure, crude:   ", round(crude.params["heart_failure"], 2), "days")
print("Heart failure, adjusted:", round(model_los.params["heart_failure"], 2), "days")

Patients with heart failure stay 1.5 days longer on average, but once age, CKD and respiratory rate are held constant the difference is 0.6 days. Patients with heart failure are older (76 vs. 68) and sicker on arrival, and that explains part of the crude difference. R² rose from 0.08 to 0.18.

## Step 3 · A yes/no outcome: in-hospital death and heart failure

In Epidemiology 1 you computed an odds ratio from a 2×2 table. Here it is again:

In [ ]:
table = pd.crosstab(df["heart_failure"], df["in_hospital_death"])
odds_hf = table.loc[1, 1] / table.loc[1, 0]
odds_no_hf = table.loc[0, 1] / table.loc[0, 0]
print(table)
print("Odds ratio from the table:", round(odds_hf / odds_no_hf, 2))

A logistic regression models the **log-odds** of death. `np.exp` of a coefficient gives an **odds ratio**, and with one binary predictor it is exactly the table's OR.

In [ ]:
model_hf = smf.logit("in_hospital_death ~ heart_failure", data=df).fit(disp=0)
print("Coefficient (log-odds):", round(model_hf.params["heart_failure"], 3))
print("Odds ratio, exp(β):    ", round(np.exp(model_hf.params["heart_failure"]), 2))

## Step 4 · Adjusted odds ratios

The same model with age, CKD, respiratory rate and SpO₂. A small helper prints each OR with its 95% confidence interval.

In [ ]:
model_death = smf.logit("in_hospital_death ~ heart_failure + age + ckd + resp_rate + spo2",
                        data=df).fit(disp=0)

ci = np.exp(model_death.conf_int())
or_table = pd.DataFrame({"OR": np.exp(model_death.params), "CI low": ci[0], "CI high": ci[1]})
or_table.drop(index="Intercept").round(2)

The heart failure OR fell from 2.78 (crude) to 1.54 (adjusted). For continuous predictors the OR is **per one unit**: per year of age, per breath per minute, per 1% of SpO₂ (below 1: higher SpO₂, lower odds).

### Your turn 2

The OR for age is per **one year**. Compute it per **10 years**. Multiply the coefficient before you exponentiate.

In [ ]:
beta_age = model_death.params["age"]
print("OR per 1 year:  ", round(np.exp(beta_age), 3))
print("OR per 10 years:", round(np.exp(____), 2))

## Step 5 · A predicted risk for every patient

`model.predict` turns the model into a probability of death for each admission. This number is what a ward team would see.

In [ ]:
df["predicted_risk"] = model_death.predict(df)
columns = ["patient_id", "age", "heart_failure", "ckd", "resp_rate", "spo2",
           "predicted_risk", "in_hospital_death"]
df.sort_values("predicted_risk", ascending=False).head(10)[columns].round(2)

The last column shows what happened. How well the ranking separates patients who died from those who survived is the subject of Data Science 3.

## Key points

- Linear regression fits a line by least squares. A coefficient is the change in the outcome per unit of the predictor; R² is the share of the variation the model explains.
- With several predictors each coefficient is **adjusted**: it compares patients alike in the other predictors.
- Logistic regression models the log-odds of a yes/no outcome. exp(β) is an odds ratio, and with one binary predictor it equals the 2×2 table OR.
- To change units, multiply β first: OR per 10 years = exp(10 × β).
- A fitted model gives every patient a predicted risk.

## Further reading

- James, Witten, Hastie, Tibshirani and Taylor. *An Introduction to Statistical Learning with Applications in Python* (ISLP), sections 3.1, 3.2.1 and 4.1–4.3. Free at [statlearning.com](https://www.statlearning.com).
- Enrichment notebook: scaling and significance in linear regression; the logit, odds ratios and the road to AUROC.